# LoRA fine-tuning on Google Colab (free T4) — fine-tune-lab

Trains a small encoder classifier with a LoRA adapter to replace the Claude
classification call in `claims-auditor` cheaply. **Runtime → Change runtime type
→ T4 GPU.** Synthetic data only. Run the cells top to bottom.

In [ ]:
# 1) Clone + install only what Colab lacks (peft). Colab already ships torch,
#    transformers, datasets, accelerate, scikit-learn — reinstalling them risks
#    version churn that breaks the GPU runtime.
!git clone https://github.com/MarcosRos002/fine-tune-lab.git
!pip -q install peft datasets accelerate scikit-learn

In [ ]:
# 2) Make the package importable WITHOUT a runtime restart (src layout).
import sys
sys.path.insert(0, '/content/fine-tune-lab/src')

from fine_tune_lab.data.build_dataset import build_dataset
from fine_tune_lab.train.config import TrainConfig
from fine_tune_lab.train.lora import train_lora, build_predictor
from fine_tune_lab.eval.metrics import evaluate
from fine_tune_lab.eval.cost import cost_per_request
from fine_tune_lab.eval.report import render_table

build_dataset(out_dir='/content/data', n=2000, seed=0)

# distilbert attention projections = q_lin / v_lin.
cfg = TrainConfig(
    base_model='distilbert-base-uncased',
    target_modules=['q_lin', 'v_lin'],
    num_epochs=5, batch_size=32, output_dir='/content/artifacts/lora',
)
train_lora('/content/data', cfg)

predict = build_predictor(cfg.output_dir, base_model=cfg.base_model)
m = evaluate(predict, '/content/data/test.jsonl')
m['cost_per_1k'] = cost_per_request(n_requests=1000, input_tokens=120000, output_tokens=20000, backend='vllm') * 1000
print('accuracy:', m['accuracy'])
print(render_table({'Fine-tuned LoRA (vLLM)': m}))